# Compare all of Parameters:
* norm_random vs. norm_subjectindipendent
* learn-rate: 1e-3 vs. 1e-4 vs. 1e-5 vs. 1e-6
* batch-size: 64 vs. 32 vs. 128
* Dataset-size(train & test): (1000 & 200) vs. (10000 & 2000) 


In [1]:
# 1: Bib

import time

import os
import re
import json
import csv
from pathlib import Path
from scipy import stats
import pandas as pd
import numpy as np
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)
from torchvision.models import (
    resnet18,
    ResNet18_Weights
)
from datetime import datetime

from torch.utils.tensorboard import SummaryWriter



I0000 00:00:1789571130.399258   14189 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789571132.112652   14189 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:

# 2: Dataset Class: transfer CSV in PyTorch.
class GazeDataset(Dataset):

    def __init__(self, csv_file, transform=None, dataset_size=None, read_all4once=True):

        self.df = pd.read_csv(csv_file)
        self.transform = transform
        self.dataset_size = dataset_size if dataset_size is not None else len(self.df)
        self.read_all4once = read_all4once

        if self.read_all4once:
            img = Image.new("RGB", (500, 300))  # any size
            out = transform(img)
            self.images = torch.zeros([self.dataset_size] + list(out.shape))
            self.targets = torch.zeros(self.dataset_size, 2)

        for idx in tqdm(range(self.dataset_size)):

            row = self.df.iloc[idx]

            image = Image.open(
                row["image_name"]
            ).convert("RGB")

            self.targets[idx] = torch.tensor(
                [row["x"], row["y"]],
                dtype=torch.float32
            )

            if self.transform:
                self.images[idx] = self.transform(image)
            else:
                self.images[idx] = image

    def __len__(self):

        return self.dataset_size

    def __getitem__(self, idx):

        return self.images[idx], self.targets[idx]

In [3]:

# 5: func-Diagonal-Error:
def diagonal_errors(model, loader, device):
    model.eval()

    targets_list = []
    preds_list = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            preds = model(images)
            targets_list.append(targets.cpu().numpy())
            preds_list.append(preds.cpu().numpy())

    targets = np.concatenate(targets_list, axis=0)
    predictions = np.concatenate(preds_list, axis=0)
    
    mae = mean_absolute_error(targets, predictions)
    
    rmse = np.sqrt(mean_squared_error(targets, predictions))

    diagonal_error_pct = np.round(100 * (rmse / np.sqrt(2)), 5)

    errors = np.sqrt(
        np.sum(
            (predictions-targets)**2,
            axis=1
        )
    )

    print(f"MAE : {mae:.4f} \t RMSE: {rmse:.4f} ")
    # print(f"RMSE: {rmse:.4f}")
    print(f"Diagonal-Error %: {diagonal_error_pct:.4f} %")

    return mae, rmse, diagonal_error_pct

In [4]:
def evaluate_model(model, loader, device):

    model.eval()

    targets_list = []
    preds_list = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            preds = model(images)
            targets_list.append(targets.cpu().numpy())
            preds_list.append(preds.cpu().numpy())

    targets = np.concatenate(targets_list, axis=0)
    predictions = np.concatenate(preds_list, axis=0)
    
    # mae = mean_absolute_error(targets, predictions)
    
    # rmse = np.sqrt(mean_squared_error(targets, predictions))

    # diagonal_error_pct = np.round(100 * (rmse / np.sqrt(2)), 5)

    errors = np.sqrt(
        np.sum(
            (predictions-targets)**2,
            axis=1
        )
    )

    # print(f"MAE : {mae:.4f} \t RMSE: {rmse:.4f} ")
    # print(f"RMSE: {rmse:.4f}")
    # print(f"Diagonal-Error %: {diagonal_error_pct:.4f} %")

    # return mae, rmse, diagonal_error_pct, targets, predictions, errors
    return targets, predictions, errors

In [5]:
class GaussianActivation(nn.Module):

    def __init__(self, sigma=1.0):
        super().__init__()
        self.sigma = sigma

    def forward(self, x):
        return torch.exp(
            -(x ** 2) / (2 * self.sigma ** 2)
        )

In [6]:
def sensitive_loss(preds, targets):

    x_total = 0.0

    for i in range(len(targets)):

        x = criterion_base(preds[i], targets[i])

        distance_from_center = (
            torch.abs(targets[i][0] - 0.5) +
            torch.abs(targets[i][1] - 0.5)
        )

        weight = 1.0 + distance_from_center

        x_total += x * weight

    return x_total / len(targets)

In [7]:
def confidence_interval(data, confidence=0.95):

    data = np.asarray(data)

    n = len(data)

    mean = np.mean(data)
    std = np.std(data, ddof=1)

    standard_error = std / np.sqrt(n)

    t_value = stats.t.ppf(
        (1 + confidence) / 2,
        df=n - 1
    )

    margin = t_value * standard_error

    lower = mean - margin
    upper = mean + margin

    return mean, lower, upper

# Hypoparameter:

* dataset_size (train-size & test-size)
* batch_size 
* learn-rate
* norm_random vs. norm_subject


In [36]:
# Hypoparameter:

name_dataset_type = ['norm_labels.csv', 'labels.csv']
dataset_size_type = [[10000, 2000], [1000, 200]]
dataset_type = ["norm_subject", "norm_random"]
batch_size_type = [32, 64, 128]
lr_type = [1e-3, 1e-4, 1e-5, 1e-6] 
epochs_num = [1, 2, 10, 25, 500]



optimizer_name = "AdamW"

# ##########################################################################
# 1.type Dataset: 'norm_labels.csv' or 'labels.csv'
dataset_name = name_dataset_type[0]                       # norm_labels.csv
# dataset_name = name_dataset_type[1]                       # labels.csv
print(f"\n dataset_name: \t {dataset_name}")


# 2.dataset_sizt: (10000 & 2000) or (1000, 200)
# def_dataset_size = dataset_size_type[0]                   # [10000, 2000]
def_dataset_size = dataset_size_type[1]                   # [1000, 200]
print(f"\n def_dataset_size: train: {def_dataset_size[0]}, \t test: {def_dataset_size[1]}")


# 3.batch_size: '32', '64' or '128'
batch_Size = batch_size_type[0]                           # 32
# batch_Size = batch_size_type[1]                           # 64
# batch_Size = batch_size_type[2]                           # 128
print(f"\n batch_Size: \t {batch_Size}")


# 4.type of dataset-split: "norm_subject_independed" or "norm_random"
def_dataset = dataset_type[0]                             # norm_subject
# def_dataset = dataset_type[1]                             # norm_random
print(f"\n def_dataset: \t {def_dataset}")


# 5.learning_rate: '1e-3', '1e-4', '1e-5' or '1e-6'
lr_type = [1e-3, 1e-4, 1e-5, 1e-6] 
# learning_rate = lr_type[0]                                # 1e-3
learning_rate = lr_type[1]                                # 1e-4
# learning_rate = lr_type[2]                                # 1e-5
# learning_rate = lr_type[3]                                # 1e-6
print(f"\n learning_rate: \t {learning_rate}")


# 6.number of epochs: 1, 2, 10, 25 or 500
epochs = epochs_num[-1]                                   # 500
print(f"\n epochs: \t {epochs}")




weight_Decay = 1e-5

active_func = None

patience = 3               # after 3 Epochen without Optimierung has to stop training
print(f"\n patience: \t {patience}")



 dataset_name: 	 norm_labels.csv

 def_dataset_size: train: 1000, 	 test: 200

 batch_Size: 	 32

 def_dataset: 	 norm_subject

 learning_rate: 	 0.0001

 epochs: 	 500

 patience: 	 3


In [ ]:
# name_dataset_type = ['norm_labels.csv', 'labels.csv']
# dataset_size_type = [[10000, 2000], [1000, 200]]
# dataset_type = ["norm_subject", "norm_random"]
# batch_size_type = [32, 64, 128]
# lr_type = [1e-3, 1e-4, 1e-5, 1e-6] 
# epochs_num = [1, 2, 10, 25, 500]

# optimizer_name = "AdamW"

# # ##########################################################################
# # 1.type Dataset: 'norm_labels.csv' or 'labels.csv'
# dataset_name = name_dataset_type[0]                       # norm_labels.csv
# # dataset_name = name_dataset_type[1]                       # labels.csv

# # 2.dataset_sizt: (10000 & 2000) or (1000, 200)
# # def_dataset_size = dataset_size_type[0]                   # [10000, 2000]
# def_dataset_size = dataset_size_type[1]                   # [1000, 200]

# # 3.batch_size: '32', '64' or '128'
# # batch_Size = batch_size_type[0]                           # 32
# # batch_Size = batch_size_type[1]                           # 64
# batch_Size = batch_size_type[2]                           # 128

# # 4.type of dataset-split: "norm_subject_independed" or "norm_random"
# def_dataset = dataset_type[0]                             # norm_subject
# # def_dataset = dataset_type[1]                             # norm_random

# # 5.learning_rate: '1e-3', '1e-4', '1e-5' or '1e-6'
# lr_type = [1e-3, 1e-4, 1e-5, 1e-6] 
# learning_rate = lr_type[0]                                # 1e-3
# # learning_rate = lr_type[1]                                # 1e-4
# # learning_rate = lr_type[2]                                # 1e-5
# # learning_rate = lr_type[3]                                # 1e-6

# epochs = epochs_num[-1]                                   # 500

# weight_Decay = 1e-5
# active_func = None
# patience = 3               # after 3 Epochen without Optimierung has to stop training

In [37]:
# 6: CSV laden
root_folder = "./dataset"
csv_path = Path(f"{root_folder}/{dataset_name}")
# print(f"\n csv_path: {csv_path}")

df = pd.read_csv(csv_path)
df.head()
# check:
# print(df.shape)

,image_name,x,y,subject_ID,screen_w,screen_h
0,dataset/images/00002/00002/frames/00000.jpg,0.500000,0.500000,2,320,568
1,dataset/images/00002/00002/frames/00001.jpg,0.500000,0.500000,2,320,568
2,dataset/images/00002/00002/frames/00002.jpg,0.500000,0.500000,2,320,568
3,dataset/images/00002/00002/frames/00003.jpg,0.500000,0.500000,2,320,568
4,dataset/images/00002/00002/frames/00004.jpg,0.873929,0.114375,2,320,568


In [38]:
# 8: DataLoader
# Transformationen:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2
    ),

    transforms.RandomGrayscale(p=0.05),

    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1,1.5)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Dataset:


if def_dataset == 'norm_subject':

    train_dataset = GazeDataset(
        "./splits/norm_subject_train.csv",
        transform, dataset_size=def_dataset_size[0],
    )
    
    test_dataset = GazeDataset(
        "./splits/norm_subject_test.csv",
        transform, dataset_size=def_dataset_size[1],
    )



elif def_dataset == 'norm_random':

    train_dataset = GazeDataset(
        "./splits/norm_random_train.csv",
        transform, dataset_size=def_dataset_size[0],
    )
    
    test_dataset = GazeDataset(
        "./splits/norm_random_test.csv",
        transform, dataset_size=def_dataset_size[1],
    )

else:

    raise ValueError(
                "The input as dataset_type is Wrong!"
            )


# Loader:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_Size,
    shuffle=True,
    num_workers=8,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_Size,
    shuffle=False,
    num_workers=8,
    persistent_workers=True
)

100%|██████████████████████████████████████████████████████████████████████████████████████████| 200/200 [00:02<00:00, 98.58it/s]


In [39]:
# 9: ResNet18 (Pretrainiertes Modell):
# Load:
def reset_model(act=None):
    model = resnet18(
        weights=ResNet18_Weights.DEFAULT
    )

    model_name = model.__class__.__name__
    
    # ---------------------------------------
    # Activation function for the last layer
    # ---------------------------------------
    if act == "Sigmoid":
        last_layer = nn.Sigmoid()

    elif act == "Gaussian":
        last_layer = GaussianActivation(sigma=1.0)

    elif act == "ReLU":
        last_layer = nn.ReLU()

    elif act == "None":
        last_layer = nn.Identity()

    else:
        raise ValueError(
            f"Unknown activation function: {act}"
        )

    # ---------------------------------------
    # Replace original ResNet FC
    # ---------------------------------------
    model.fc = nn.Sequential(
        nn.Linear(
            model.fc.in_features,
            512
        ),
        nn.ReLU(),

        nn.Linear(
            512,
            256
        ),
        nn.ReLU(),

        nn.Dropout(0.2),

        nn.Linear(
            256,
            128
        ),
        nn.ReLU(),

        nn.Linear(
            128,
            2
        ),

        # Last activation
        last_layer
    )

    return model, model_name

In [40]:
# 10: GPU or CPU
# check:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [32]:

print(f"\n\t dataset_name: \t {dataset_name}")
print(f"\t def_dataset_size: train: {def_dataset_size[0]}, \t test: {def_dataset_size[1]}")
print(f"\t batch_Size: \t {batch_Size}")
print(f"\t def_dataset: \t {def_dataset}")
print(f"\t learning_rate: \t {learning_rate}")
print(f"\t epochs: \t {epochs}")
print(f"\t patience: \t {patience}")



parm1, parm2 = [], []

parm_test_error = []
parm_run_time = []




criterion = nn.L1Loss()

criterion_base = nn.L1Loss()

epochs = epochs

if globals().get("bas_const_err") is None:
    bas_const_err = 27.245

if globals().get("bas_rand_err") is None:
    bas_rand_err = 38.961
    

model_output = './models/best_models'
model_dir= Path(model_output)

if not os.path.exists(model_dir):
    model_dir.mkdir(
        parents=True,
        exist_ok=True
    )


diagrams_output = './results/best_diagrams'
diagrams_dir= Path(diagrams_output)

if not os.path.exists(diagrams_dir):
    diagrams_dir.mkdir(
        parents=True,
        exist_ok=True
    )

diag_test_errors = {}
diag_train_errors = {}


act_time_epochs = {}


diag_test_error, diag_train_error = [], []

acts = ['Sigmoid', 'Gaussian', 'ReLU', 'None']

act = acts[0]

# for act in ['Sigmoid', 'Gaussian', 'ReLU', 'None']:
# for act in ['Sigmoid']:
# for act in ['None']:

if act == 'Sigmoid':

    train_start = time.perf_counter()
  
    model, model_name = reset_model(act=act)

    active_func = act

    diag_test_errors[act] = []
    diag_train_errors[act] = []

    best_error = float("inf")

    t = 0                      # Number Epochen without Optimierung
    e = 0                      # Number Epochen with Optimierung


    model.to(device)



    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the head
    for param in model.layer4.parameters():
        param.requires_grad=False

    for param in model.fc.parameters():
        param.requires_grad=True

    optimizer = torch.optim.AdamW(
        filter(lambda p:p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=weight_Decay
    )

    print('\n ',"=#=" * 25)
    print(f"\t\t act: {act}")
    print(' ',"=#=" * 25)

    print("\n\t Start: \n")
    
    print("train_error:")
    train_mae, train_rmse, train_diag_pct = diagonal_errors(model, train_loader, device)
    diag_train_error.append(np.round(train_diag_pct, 4))
    
    diag_train_errors[act].append(np.round(train_diag_pct, 4))



    print("\ntest_error:")
    test_mae, test_rmse, test_diag_pct = diagonal_errors(model, test_loader, device)
    diag_test_error.append(np.round(test_diag_pct, 4))

    diag_test_errors[act].append(np.round(test_diag_pct, 4))

    if best_error is None or test_diag_pct < best_error:    
        # set the first test-error
        best_error = test_diag_pct
        t = 0
        e += 1


    print("\n\n\t Training: \n")

    for epoch in range(epochs):

        epoch_start = time.perf_counter()

        model.train()
        running_loss = 0

        loop = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}"
        )

        for images, targets in loop:
            images = images.to(device)
            targets = targets.to(device)
            optimizer.zero_grad()
            preds = model(images)

            ################################
            
            # Loss 1:
            loss = criterion(
                preds,
                targets
            )

            
            # Loss 2: 
            # loss = sensitive_loss(
            #     preds,
            #     targets
            # )
            
            ################################

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            loop.set_postfix(
                loss=loss.item()
            )

        # epoch_end = time.perf_counter()

        print(f"\ntrain_error:")
        train_mae, train_rmse, train_diag_pct = diagonal_errors(model, train_loader, device)
        diag_train_error.append(np.round(train_diag_pct, 4))

        diag_train_errors[act].append(np.round(train_diag_pct, 4))

        print(f"\ntest_error:")
        test_mae, test_rmse, test_diag_pct = diagonal_errors(model, test_loader, device)
        diag_test_error.append(np.round(test_diag_pct, 4))

        diag_test_errors[act].append(np.round(test_diag_pct, 4))

        # --------------------------------------------------
        # Early Stopping
        # --------------------------------------------------
        
        if best_error is None or test_diag_pct < best_error:
        
            # find an improvement
            best_error = test_diag_pct
            t = 0
            e += 1
        
            print(
                f"Test-Diagonal-Error: {test_diag_pct:.4f}% | "
                f"Improvment: {e}"
            )
        
            # save the better Modell
            torch.save(
                model.state_dict(),
                f"./models/best_models/"
                f"best_{model_name}_{act}.path"
            )

            
            # torch.save(
            #     model.state_dict(),
            #     f"./models/best_optim-model_"
            #     f"{def_dataset}_{def_dataset_size[0]}-{def_dataset_size[1]}.path"
            # )

        
        else:
        
            # without imporovement
            t += 1
        
            print(
                f"Patience: {t}/{patience}"
            )
        
            if t >= patience:
                print(
                    f"\nEarly Stopping after {epoch + 1} Epochen."
                )
                print(
                    f"Imporovments totally: {e}"
                )
                break



        epoch_end = time.perf_counter()


        print(
            f"\n[{datetime.now().strftime('%H:%M:%S')}] Epoch {epoch + 1}: {epoch_end - epoch_start:.2f} Sekunden | "
            f"Running_loss: {running_loss / len(train_loader):.3f} | "
            f"test_diag_error={test_diag_pct:.4f}% | "
            f"train_diag_error={train_diag_pct:.4f}% | \n"
        )

        # torch.save(model.state_dict(),
        #         f"./models/last_best_optim_model_{def_dataset}_{def_dataset_size[0]}-{def_dataset_size[1]}.path")

        torch.save(
                model.state_dict(),
                f"./models/best_models/"
                f"last_{model_name}_{act}.path"
            )


    train_end = time.perf_counter()


    elapsed_running_time = train_end - train_start
    print(f"\ntotll running-tiems (s): {elapsed_running_time:.2f} Sekunden")
    print(f"totll running-tiems (min): {elapsed_running_time / 60:.2f} Minuten\n")


    act_time_epochs[act] = {
        "time_minutes": elapsed_running_time / 60,
        "epochs": epoch + 1,
        "improvements": e,
        "best_test_error": best_error
    }


    parm1.append(best_error)
    parm2.append(elapsed_running_time / 60)

print(f"\n\n Parameter:\n best_test_error: {np.float64(parm1)}, \n running_time: {np.float64(parm2)} \n")





	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 32
	 def_dataset: 	 norm_subject
	 learning_rate: 	 0.0001
	 epochs: 	 500
	 patience: 	 3
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.fc.parameters: True
param in model.fc.parameters: True
param in model.fc.parameters: True
param in model.fc.parameters: True
param in model.fc.parameters: True
param in model.fc.parameters: True
par

Epoch 1: 100%|███████████████████████████████████████████████████████████████████████| 32/32 [00:47<00:00,  1.50s/it, loss=0.211]



train_error:
MAE : 0.2291 	 RMSE: 0.2766 
Diagonal-Error %: 19.5593 %

test_error:
MAE : 0.2364 	 RMSE: 0.2843 
Diagonal-Error %: 20.1041 %
Test-Diagonal-Error: 20.1041% | Improvment: 2

[13:49:36] Epoch 1: 105.75 Sekunden | Running_loss: 0.231 | test_diag_error=20.1041% | train_diag_error=19.5593% | 



Epoch 2: 100%|███████████████████████████████████████████████████████████████████████| 32/32 [00:51<00:00,  1.61s/it, loss=0.261]



train_error:
MAE : 0.2264 	 RMSE: 0.2743 
Diagonal-Error %: 19.3963 %

test_error:
MAE : 0.2413 	 RMSE: 0.2885 
Diagonal-Error %: 20.4009 %
Patience: 1/3

[13:51:27] Epoch 2: 110.78 Sekunden | Running_loss: 0.230 | test_diag_error=20.4009% | train_diag_error=19.3963% | 



Epoch 3: 100%|███████████████████████████████████████████████████████████████████████| 32/32 [00:53<00:00,  1.67s/it, loss=0.204]



train_error:
MAE : 0.2243 	 RMSE: 0.2724 
Diagonal-Error %: 19.2643 %

test_error:
MAE : 0.2440 	 RMSE: 0.2909 
Diagonal-Error %: 20.5724 %
Patience: 2/3

[13:53:23] Epoch 3: 115.92 Sekunden | Running_loss: 0.226 | test_diag_error=20.5724% | train_diag_error=19.2643% | 



Epoch 4: 100%|███████████████████████████████████████████████████████████████████████| 32/32 [01:04<00:00,  2.03s/it, loss=0.222]



train_error:
MAE : 0.2229 	 RMSE: 0.2722 
Diagonal-Error %: 19.2453 %

test_error:
MAE : 0.2547 	 RMSE: 0.3029 
Diagonal-Error %: 21.4151 %
Patience: 3/3

Early Stopping after 4 Epochen.
Imporovments totally: 2

totll running-tiems (s): 519.24 Sekunden
totll running-tiems (min): 8.65 Minuten



 Parameter:
 best_test_error: [20.10409], 
 running_time: [8.65406785] 



In [33]:
for act, values in act_time_epochs.items():
    print(
        f"last_layer: {act}\t"
        f" epochs: {values['epochs']} Epochen\t|"
        f" running_time: {values['time_minutes']:.2f} min\t |"
        f" improvements: {values['improvements']} Epochen\t |"
        f" best_test_error: {values['best_test_error']:.4f} % "
    )

last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 8.65 min	 | improvements: 2 Epochen	 | best_test_error: 20.1041 % 


In [ ]:

print(f"\n\n Diagonal-Train-Error: {np.float64(diag_train_error)}\n")
print(f" Diagonal-Test-Error: {np.float64(diag_test_error)}\n\n")

print(f" Diagonal-Train-Error-Max: {np.max(diag_train_error)} \t Diagonal-Train-Error-Min: {np.min(diag_train_error)}\t")
# print(f" Diagonal-Train-Error-Ave: {np.mean(diag_train_error)}\n")

print(f" Diagonal-Test-Error-Max: {np.max(diag_test_error)}\t Diagonal-Test-Error-Min: {np.min(diag_test_error)}\t max_needed_epochs: {e+2}\n\n")
# print(f" Diagonal-Test-Error-Ave: {np.mean(diag_test_error)}\n")


if globals().get("bas_const_err") is None:
    bas_const_err = 27.245

if globals().get("bas_rand_err") is None:
    bas_rand_err = 38.961


# Curven:

# Minimum Test Error bestimmen
min_test_error = min(diag_test_error)
min_test_epoch = diag_test_error.index(min_test_error) + 1

# Die vier Werte an dieser Epoch
train_at_min = diag_train_error[min_test_epoch - 1]
test_at_min = diag_test_error[min_test_epoch - 1]
const_at_min = bas_const_err
rand_at_min = bas_rand_err

# Die Farbe
train_color = "tab:blue"
test_color = "tab:orange"
const_color = "tab:green"
rand_color = "tab:red"
point_color = "black"

# Epochs beginnen bei 1
epochs = range(1, len(diag_test_error) + 1)
# epochs = range(0, len(diag_test_error) + 1)

plt.figure(figsize=(10, 7))

# Train Error
plt.plot(epochs, diag_train_error, label="Train", linewidth=2, color=train_color)


# Test Error
plt.plot(epochs, diag_test_error, label="Test", linewidth=2, color=test_color)


# Vertikale Linie durch alle vier Kurven
plt.axvline(
    x=min_test_epoch,
    linestyle="--",
    linewidth=1.2,
    alpha=0.7,
    # label=f"Min Test Error (Epoch {min_test_epoch})"
)

# Punkt beim Minimum des Test Errors
plt.scatter(
    min_test_epoch,
    min_test_error,
    s=40,
    zorder=5,
    color=point_color
    # label=f"Min Test Error ({min_test_error:.2f}%)"
)

plt.scatter(
    min_test_epoch,
    train_at_min,
    s=40,
    zorder=5,
    color=point_color
    # label=f"Train at min_Test_Error ({train_at_min_test:.2f}%)"
)

# Werte an der vertikalen Linie anzeigen
plt.annotate(
    f"'Train': {train_at_min:.3f}%",
    xy=(min_test_epoch, train_at_min),
    xytext=(8, 10),
    textcoords="offset points",
    color=train_color,
    fontsize=8
)

plt.annotate(
    f"'Test': {test_at_min:.3f}%",
    xy=(min_test_epoch, test_at_min),
    xytext=(8, -10),
    textcoords="offset points",
    color=test_color,
    fontsize=8,
)


plt.title("Train- vs. Test-Error:")
plt.xlabel("Epoche")
plt.ylabel("Diagnostic Error (%)")

plt.xticks(epochs)

plt.grid(True, alpha=0.3, linewidth=0.6)
plt.legend()



# ===================================================
# =============== all 4 Diagrams ====================
# ===================================================


plt.figure(figsize=(10, 7))


# Train Error
plt.plot(
    epochs,
    diag_train_error,
    label="Train",
    linewidth=2,
    color=train_color
)


# Test Error
plt.plot(
    epochs,
    diag_test_error,
    label="Test",
    linewidth=2,
    color=test_color
)


# Constant Baseline
plt.plot(
    epochs,
    [bas_const_err] * len(epochs),
    label="Constant Error",
    linestyle="--",
    linewidth=1.2,
    alpha=0.6,
    color=const_color
)


# Random Baseline
plt.plot(
    epochs,
    [bas_rand_err] * len(epochs),
    label="Random Error",
    linestyle="-.",
    linewidth=1.2,
    alpha=0.6,
    color=rand_color
)



print(f"\nMinimum Test Error:")
print(f"Epoch:           {min_test_epoch}")
print(f"Test Error:      {test_at_min:.4f}%")
print(f"Train Error:     {train_at_min:.4f}%")
print(f"Constant Error:  {bas_const_err:.4f}%")
print(f"Random Error:    {bas_rand_err:.4f}%\n")


# Punkt beim Minimum des Test Errors
plt.scatter(
    min_test_epoch,
    min_test_error,
    s=40,
    zorder=5,
    color=point_color
    # label=f"Min Test Error ({min_test_error:.2f}%)"
)

plt.scatter(
    min_test_epoch,
    train_at_min,
    s=40,
    zorder=5,
    color=point_color
    # label=f"Train at min_Test_Error ({train_at_min_test:.2f}%)"
)

plt.scatter(
    min_test_epoch,
    const_at_min,
    s=20,
    zorder=4,
    color=point_color
)

plt.scatter(
    min_test_epoch,
    rand_at_min,
    s=20,
    zorder=4,
    color=point_color
)


# Vertikale Linie durch alle vier Kurven
plt.axvline(
    x=min_test_epoch,
    linestyle="--",
    linewidth=1.2,
    alpha=0.7,
    # label=f"Min Test Error (Epoch {min_test_epoch})"
)

# Werte an der vertikalen Linie anzeigen
plt.annotate(
    f"'Train': {train_at_min:.3f}%",
    xy=(min_test_epoch, train_at_min),
    xytext=(5, 5),
    textcoords="offset points",
    color=train_color,
    fontsize=8
)

plt.annotate(
    f"'Test': {test_at_min:.3f}%",
    xy=(min_test_epoch, test_at_min),
    xytext=(5, 7),
    textcoords="offset points",
    color=test_color,
    fontsize=8
)

plt.annotate(
    f"'Constant': {const_at_min:.3f}%",
    xy=(min_test_epoch, const_at_min),
    xytext=(5, 10),
    textcoords="offset points",
    color=const_color,
    fontsize=7
)

plt.annotate(
    f"'Random': {rand_at_min:.3f}%",
    xy=(min_test_epoch, rand_at_min),
    xytext=(5, -15),
    textcoords="offset points",
    color=rand_color,
    fontsize=7
)

# Titel
plt.title(
    f"{def_dataset} "
    f"(train:{def_dataset_size[0]}, test:{def_dataset_size[1]})\n"
    f"lr = {learning_rate} | "
    f"{elapsed_running_time / 60:.2f} Minuten | "
    f"extra_act_fun: {active_func} | "
    f"epochs_improvements_totally: {e + 2}"
)
# plt.title(f"{def_dataset}((train:{def_dataset_size[0]}, test:{def_dataset_size[1]})) \n" 
#           f" lr = {learning_rate} | {elapsed_running_time/60:.2f} Minuten | extra_act_fun: {active_func} | epochs_imporovments_totally: {e+2}"
# )

plt.xlabel("Epoche")
plt.ylabel("Diagnostic Error (%)")

plt.xticks(epochs)

# plt.xticks(range(1, len(epochs) + 1, 1))
# plt.yticks(range(0, 51, 5))

plt.grid(True, alpha=0.4)
plt.legend()

# Ordner erstellen (falls er noch nicht existiert)
os.makedirs("./results", exist_ok=True)

# Diagramm speichern
plt.savefig(
    f"results/diag_error_{def_dataset}_{learning_rate}_{def_dataset_size[0]}_{def_dataset_size[1]}_a.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

    
    
print(f"\n\t dataset_name: \t {dataset_name}")
print(f"\t def_dataset_size: train: {def_dataset_size[0]}, \t test: {def_dataset_size[1]}")
print(f"\t batch_Size: \t {batch_Size}")
print(f"\t def_dataset: \t {def_dataset}")
print(f"\t learning_rate: \t {learning_rate}")
print(f"\t epochs: \t {epochs}")
print(f"\t patience: \t {patience}")


    
parm1, parm2 = [], []

parm_test_error = []
parm_run_time = []


criterion = nn.L1Loss()

criterion_base = nn.L1Loss()

epochs = epochs

if globals().get("bas_const_err") is None:
    bas_const_err = 30.987 

if globals().get("bas_rand_err") is None:
    bas_rand_err  = 42.702 
    


diag_test_errors = {}
diag_train_errors = {}


act_time_epochs = {}


diag_test_error, diag_train_error = [], []

acts = ['Sigmoid', 'Gaussian', 'ReLU', 'None']

for i in range(len(acts)):

    act = acts[i]
    
    for t in range(5):
        
        print(f"\n\t t: {t + 1}")
        

        
        
        train_start = time.perf_counter()
      
        model, model_name = reset_model(act=act)
    
        active_func = act
    
        diag_test_errors[act] = []
        diag_train_errors[act] = []
    
        best_error = float("inf")
    
        t = 0                      # Number Epochen without Optimierung
        e = 0                      # Number Epochen with Optimierung
    
    
        model.to(device)
    
    
    
        for param in model.parameters():
            param.requires_grad = False
    
        # Unfreeze the head
        for param in model.layer4.parameters():
            param.requires_grad=True
    
        for param in model.fc.parameters():
            param.requires_grad=True
    
    
        optimizer = torch.optim.AdamW(
            filter(lambda p:p.requires_grad, model.parameters()),
            lr=learning_rate,
            weight_decay=weight_Decay
        )
    
        print('\n ',"=#=" * 25)
        print(f"\t\t act: {act}")
        print(' ',"=#=" * 25)
    
        print("\n\t Start: \n")
        
        print("train_error:")
        train_mae, train_rmse, train_diag_pct = diagonal_errors(model, train_loader, device)
        diag_train_error.append(np.round(train_diag_pct, 4))
        
        diag_train_errors[act].append(np.round(train_diag_pct, 4))
    
    
    
        print("\ntest_error:")
        test_mae, test_rmse, test_diag_pct = diagonal_errors(model, test_loader, device)
        diag_test_error.append(np.round(test_diag_pct, 4))
    
        diag_test_errors[act].append(np.round(test_diag_pct, 4))

        if best_error is None or test_diag_pct < best_error:    
            # set the first test-error
            best_error = test_diag_pct
            t = 0
            e += 1
    
    
        print("\n\n\t Training: \n")
    
        for epoch in range(epochs):
    
            epoch_start = time.perf_counter()
    
            model.train()
            running_loss = 0
    
            loop = tqdm(
                train_loader,
                desc=f"Epoch {epoch + 1}"
            )
    
            for images, targets in loop:
                images = images.to(device)
                targets = targets.to(device)
                optimizer.zero_grad()
                preds = model(images)
    
                ################################
                
                # Loss 1:
                loss = criterion(
                    preds,
                    targets
                )
    
                
                ################################
    
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
    
                loop.set_postfix(
                    loss=loss.item()
                )
    
            # epoch_end = time.perf_counter()
    
            print(f"\ntrain_error:")
            train_mae, train_rmse, train_diag_pct = diagonal_errors(model, train_loader, device)
            diag_train_error.append(np.round(train_diag_pct, 4))
    
            diag_train_errors[act].append(np.round(train_diag_pct, 4))
    
            print(f"\ntest_error:")
            test_mae, test_rmse, test_diag_pct = diagonal_errors(model, test_loader, device)
            diag_test_error.append(np.round(test_diag_pct, 4))
    
            diag_test_errors[act].append(np.round(test_diag_pct, 4))
    
            # --------------------------------------------------
            # Early Stopping
            # --------------------------------------------------
            
            if best_error is None or test_diag_pct < best_error:
            
                # find an improvement
                best_error = test_diag_pct
                t = 0
                e += 1
            
                print(
                    f"Test-Diagonal-Error: {test_diag_pct:.4f}% | "
                    f"Improvment: {e}"
                )
            
            
            else:
            
                # without imporovement
                t += 1
            
                print(
                    f"Patience: {t}/{patience}"
                )
            
                if t >= patience:
                    print(
                        f"\nEarly Stopping after {epoch + 1} Epochen."
                    )
                    print(
                        f"Imporovments totally: {e}"
                    )
                    break
    
    
    
            epoch_end = time.perf_counter()
    
    
            print(
                f"\n[{datetime.now().strftime('%H:%M:%S')}] Epoch {epoch + 1}: {epoch_end - epoch_start:.2f} Sekunden | "
                f"Running_loss: {running_loss / len(train_loader):.3f} | "
                f"test_diag_error={test_diag_pct:.4f}% | "
                f"train_diag_error={train_diag_pct:.4f}% | \n"
            )
    
    
        train_end = time.perf_counter()
    
    
        elapsed_running_time = train_end - train_start
        print(f"\ntotll running-tiems (s): {elapsed_running_time:.2f} Sekunden")
        print(f"totll running-tiems (min): {elapsed_running_time / 60:.2f} Minuten\n")
    
    
        act_time_epochs[act] = {
            "time_minutes": elapsed_running_time / 60,
            "epochs": epoch + 1,
            "improvements": e,
            "best_test_error": best_error
        }
    
    
        parm1.append(best_error)
        parm2.append(elapsed_running_time / 60)

    
    
    
        for act, values in act_time_epochs.items():
            print(
                f"last_layer: {act}\t"
                f" epochs: {values['epochs']} Epochen\t|"
                f" running_time: {values['time_minutes']:.2f} min\t |"
                f" improvements: {values['improvements']} Epochen\t |"
                f" best_test_error: {values['best_test_error']:.4f} % "
            )

    
    print('\n ',"=#=" * 25)
    print(' ',"=#=" * 25)
    

    print(f"\n\n Parameter:\n best_test_error: {np.float64(parm1)}, \n running_time: {np.float64(parm2)} \n")



	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 32
	 def_dataset: 	 norm_subject
	 learning_rate: 	 0.0001
	 epochs: 	 500
	 patience: 	 3

	 t: 1

  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=
		 act: Sigmoid
  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=

	 Start: 

train_error:
MAE : 0.2318 	 RMSE: 0.2792 
Diagonal-Error %: 19.7432 %

test_error:
MAE : 0.2347 	 RMSE: 0.2829 
Diagonal-Error %: 20.0008 %


	 Training: 



Epoch 1: 100%|████████████████████████████████████████████████████████████████████████| 32/32 [01:20<00:00,  2.51s/it, loss=0.32]



train_error:
MAE : 0.2200 	 RMSE: 0.2684 
Diagonal-Error %: 18.9759 %

test_error:
MAE : 0.2433 	 RMSE: 0.2904 
Diagonal-Error %: 20.5309 %
Patience: 1/3

[14:26:36] Epoch 1: 135.42 Sekunden | Running_loss: 0.230 | test_diag_error=20.5309% | train_diag_error=18.9759% | 



Epoch 2: 100%|███████████████████████████████████████████████████████████████████████| 32/32 [01:08<00:00,  2.13s/it, loss=0.168]



train_error:
MAE : 0.2033 	 RMSE: 0.2577 
Diagonal-Error %: 18.2199 %

test_error:
MAE : 0.2559 	 RMSE: 0.3047 
Diagonal-Error %: 21.5451 %
Patience: 2/3

[14:28:39] Epoch 2: 123.21 Sekunden | Running_loss: 0.216 | test_diag_error=21.5451% | train_diag_error=18.2199% | 



...


train_error:
MAE : 0.1662 	 RMSE: 0.2161 
Diagonal-Error %: 15.2806 %

test_error:
MAE : 0.3030 	 RMSE: 0.3660 
Diagonal-Error %: 25.8803 %
Patience: 3/3

Early Stopping after 5 Epochen.
Imporovments totally: 3

totll running-tiems (s): 893.20 Sekunden
totll running-tiems (min): 14.89 Minuten

last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 7.86 min	 | improvements: 1 Epochen	 | best_test_error: 19.9700 % 
last_layer: Gaussian	 epochs: 5 Epochen	| running_time: 18.34 min	 | improvements: 3 Epochen	 | best_test_error: 21.5474 % 
last_layer: ReLU	 epochs: 5 Epochen	| running_time: 14.89 min	 | improvements: 3 Epochen	 | best_test_error: 20.3017 % 

  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=
  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=


 Parameter:
 best_test_error: [20.00081 20.06476 20.31059 20.2817  19.97003 20.83019 20.97602 22.25679
 22.18371 21.54738 30.8811  20.24064 20.35178 31.05322 20.3017 ], 
 running_

Epoch 1: 100%|███████████████████████████████████████████████████████████████████████| 32/32 [01:35<00:00,  2.97s/it, loss=0.167]



train_error:
MAE : 0.2175 	 RMSE: 0.2711 
Diagonal-Error %: 19.1664 %

test_error:
MAE : 0.2430 	 RMSE: 0.2887 
Diagonal-Error %: 20.4125 %
Test-Diagonal-Error: 20.4125% | Improvment: 2

[18:06:16] Epoch 1: 171.28 Sekunden | Running_loss: 0.260 | test_diag_error=20.4125% | train_diag_error=19.1664% | 



Epoch 2: 100%|███████████████████████████████████████████████████████████████████████| 32/32 [01:26<00:00,  2.72s/it, loss=0.164]



train_error:


	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 32
	 def_dataset: 	 norm_subject
	 learning_rate: 	 0.0001
	 epochs: 	 500
	 patience: 	 3
     Layer 4 : False
     FC : True

# Sigmoid :
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 7.59 min	 | improvements: 1 Epochen	 | best_test_error: 20.0008 % 
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 7.00 min	 | improvements: 1 Epochen	 | best_test_error: 20.0648 % 
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 7.46 min	 | improvements: 1 Epochen	 | best_test_error: 20.3106 % 
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 8.68 min	 | improvements: 2 Epochen	 | best_test_error: 20.2817 %
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 7.86 min	 | improvements: 1 Epochen	 | best_test_error: 19.9700 %

best_test_error1 = [20.00081, 20.06476, 20.31059, 20.2817, 19.97003]
run_time1 =        [7.594, 7.003, 7.463, 8.684, 7.862]


# Gaussien :
last_layer: Gaussian  epochs: 8 Epochen	| running_time: 17.84 min	 | improvements: 4 Epochen	 | best_test_error: 20.8302 %
last_layer: Gaussian  epochs: 6 Epochen	| running_time: 24.44 min	 | improvements: 4 Epochen	 | best_test_error: 20.9760 % 
last_layer: Gaussian  epochs: 6 Epochen	| running_time: 26.45 min	 | improvements: 4 Epochen	 | best_test_error: 22.2568 %
last_layer: Gaussian  epochs: 6 Epochen	| running_time: 25.15 min	 | improvements: 4 Epochen	 | best_test_error: 22.1837 %


best_test_error2 = []
run_time2 = []

# ReLU :

best_test_error3 = []
run_time3 = []

# None:

best_test_error4 = []
run_time4 = []

In [24]:

print(f"\n\t dataset_name: \t {dataset_name}")
print(f"\t def_dataset_size: train: {def_dataset_size[0]}, \t test: {def_dataset_size[1]}")
print(f"\t batch_Size: \t {batch_Size}")
print(f"\t def_dataset: \t {def_dataset}")
print(f"\t learning_rate: \t {learning_rate}")
print(f"\t epochs: \t {epochs}")
print(f"\t patience: \t {patience}")



parm1, parm2 = [], []

parm_test_error = []
parm_run_time = []



for t in range(5):
    
    print(f"\n\t t: {t + 1}")
    
    criterion = nn.L1Loss()
    
    criterion_base = nn.L1Loss()
    
    epochs = epochs
    
    if globals().get("bas_const_err") is None:
        bas_const_err = 27.245
    
    if globals().get("bas_rand_err") is None:
        bas_rand_err = 38.961
        
    
    # model_output = './models'
    # model_dir= Path(model_output)
    
    # if not os.path.exists(model_dir):
    #     model_dir.mkdir(
    #         parents=True,
    #         exist_ok=True
    #     )
    
    
    # diagrams_output = './results/diagrams'
    # diagrams_dir= Path(diagrams_output)
    
    # if not os.path.exists(diagrams_dir):
    #     diagrams_dir.mkdir(
    #         parents=True,
    #         exist_ok=True
    #     )
    
    diag_test_errors = {}
    diag_train_errors = {}
    
    
    act_time_epochs = {}
    
    
    diag_test_error, diag_train_error = [], []
    
    acts = ['Sigmoid', 'Gaussian', 'ReLU', 'None']
    
    act = acts[0]
    
    # for act in ['Sigmoid', 'Gaussian', 'ReLU', 'None']:
    # for act in ['Sigmoid']:
    # for act in ['None']:
    
    if act == 'Sigmoid':
    
    # if act == 'None':
    
        train_start = time.perf_counter()
      
        model, model_name = reset_model(act=act)
    
        active_func = act
    
        diag_test_errors[act] = []
        diag_train_errors[act] = []
    
        best_error = float("inf")
    
        t = 0                      # Number Epochen without Optimierung
        e = 0                      # Number Epochen with Optimierung
    
    
        model.to(device)
    
    
    
        for param in model.parameters():
            param.requires_grad = False
    
        # Unfreeze the head
        for param in model.layer4.parameters():
            param.requires_grad=False
            print(f"param in model.layer4.parameters: {param.requires_grad}")
    
        for param in model.fc.parameters():
            param.requires_grad=True
            print(f"param in model.fc.parameters: {param.requires_grad}")
    
    
        optimizer = torch.optim.AdamW(
            filter(lambda p:p.requires_grad, model.parameters()),
            lr=learning_rate,
            weight_decay=weight_Decay
        )
    
        print('\n ',"=#=" * 25)
        print(f"\t\t act: {act}")
        print(' ',"=#=" * 25)
    
        print("\n\t Start: \n")
        
        print("train_error:")
        train_mae, train_rmse, train_diag_pct = diagonal_errors(model, train_loader, device)
        diag_train_error.append(np.round(train_diag_pct, 4))
        
        diag_train_errors[act].append(np.round(train_diag_pct, 4))
    
    
    
        print("\ntest_error:")
        test_mae, test_rmse, test_diag_pct = diagonal_errors(model, test_loader, device)
        diag_test_error.append(np.round(test_diag_pct, 4))
    
        diag_test_errors[act].append(np.round(test_diag_pct, 4))

        if best_error is None or test_diag_pct < best_error:    
            # set the first test-error
            best_error = test_diag_pct
            t = 0
            e += 1
    
    
        print("\n\n\t Training: \n")
    
        for epoch in range(epochs):
    
            epoch_start = time.perf_counter()
    
            model.train()
            running_loss = 0
    
            loop = tqdm(
                train_loader,
                desc=f"Epoch {epoch + 1}"
            )
    
            for images, targets in loop:
                images = images.to(device)
                targets = targets.to(device)
                optimizer.zero_grad()
                preds = model(images)
    
                ################################
                
                # Loss 1:
                loss = criterion(
                    preds,
                    targets
                )
    
                
                # Loss 2: 
                # loss = sensitive_loss(
                #     preds,
                #     targets
                # )
                
                ################################
    
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
    
                loop.set_postfix(
                    loss=loss.item()
                )
    
            # epoch_end = time.perf_counter()
    
            print(f"\ntrain_error:")
            train_mae, train_rmse, train_diag_pct = diagonal_errors(model, train_loader, device)
            diag_train_error.append(np.round(train_diag_pct, 4))
    
            diag_train_errors[act].append(np.round(train_diag_pct, 4))
    
            print(f"\ntest_error:")
            test_mae, test_rmse, test_diag_pct = diagonal_errors(model, test_loader, device)
            diag_test_error.append(np.round(test_diag_pct, 4))
    
            diag_test_errors[act].append(np.round(test_diag_pct, 4))
    
            # --------------------------------------------------
            # Early Stopping
            # --------------------------------------------------
            
            if best_error is None or test_diag_pct < best_error:
            
                # find an improvement
                best_error = test_diag_pct
                t = 0
                e += 1
            
                print(
                    f"Test-Diagonal-Error: {test_diag_pct:.4f}% | "
                    f"Improvment: {e}"
                )
            
                # save the better Modell
                # torch.save(
                #     model.state_dict(),
                #     f"./models/best_optim-model_"
                #     f"{def_dataset}_{def_dataset_size[0]}-{def_dataset_size[1]}.path"
                # )
    
                # torch.save(
                #     model.state_dict(),
                #     f"./models/best-model_"
                #     f"{model_name}_{act}.path"
                # )
            
            else:
            
                # without imporovement
                t += 1
            
                print(
                    f"Patience: {t}/{patience}"
                )
            
                if t >= patience:
                    print(
                        f"\nEarly Stopping after {epoch + 1} Epochen."
                    )
                    print(
                        f"Imporovments totally: {e}"
                    )
                    break
    
    
    
            epoch_end = time.perf_counter()
    
    
            print(
                f"\n[{datetime.now().strftime('%H:%M:%S')}] Epoch {epoch + 1}: {epoch_end - epoch_start:.2f} Sekunden | "
                f"Running_loss: {running_loss / len(train_loader):.3f} | "
                f"test_diag_error={test_diag_pct:.4f}% | "
                f"train_diag_error={train_diag_pct:.4f}% | \n"
            )
    
            # torch.save(model.state_dict(),
            #         f"./models/last_best_optim_model_{def_dataset}_{def_dataset_size[0]}-{def_dataset_size[1]}.path")
    
            # torch.save(
            #         model.state_dict(),
            #         f"./models/last-model_"
            #         f"{model_name}_{act}.path"
            #     )
    
    
        train_end = time.perf_counter()
    
    
        elapsed_running_time = train_end - train_start
        print(f"\ntotll running-tiems (s): {elapsed_running_time:.2f} Sekunden")
        print(f"totll running-tiems (min): {elapsed_running_time / 60:.2f} Minuten\n")
    
        
        # act_time[act].append(elapsed_running_time)
        # act_epoch[act].append(e)
    
        act_time_epochs[act] = {
            "time_minutes": elapsed_running_time / 60,
            "epochs": epoch + 1,
            "improvements": e,
            "best_test_error": best_error
        }
    
    
        parm1.append(best_error)
        parm2.append(elapsed_running_time / 60)

    
        # print(f"\n\nDiagonal_train_error: {np.float64(diag_train_error)}")
        # print(f"\n\nDiagonal_test_error: {np.float64(diag_test_error)}")

    for act, values in act_time_epochs.items():
        print(
            f"last_layer: {act}\t"
            f" epochs: {values['epochs']} Epochen\t|"
            f" running_time: {values['time_minutes']:.2f} min\t |"
            f" improvements: {values['improvements']} Epochen\t |"
            f" best_test_error: {values['best_test_error']:.4f} % "
        )
        # parm_test_error.append(float(f"{values['best_test_error']:.4f}"))
        # parm_run_time.append(float(f"{values['time_minutes']:.2f}"))

    print('\n ',"=#=" * 25)
    print(' ',"=#=" * 25)

# print(f"\n\n Parameter:\n best_test_error: {param_test_error}, \n running_time: {parm_run_time} \n")
print(f"\n\n Parameter:\n best_test_error: {np.float64(parm1)}, \n running_time: {np.float64(parm2)} \n")



	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 10000, 	 test: 2000
	 batch_Size: 	 32
	 def_dataset: 	 norm_subject
	 learning_rate: 	 0.0001
	 epochs: 	 500
	 patience: 	 3

	 t: 1
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.layer4.parameters: False
param in model.fc.parameters: True
param in model.fc.parameters: True
param in model.fc.parameters: True
param in model.fc.parameters: True
param in model.fc.parameters: True
param in model.fc.parameters

Epoch 1: 100%|█████████████████████████████████████████████████████████████████████| 313/313 [08:31<00:00,  1.63s/it, loss=0.235]



train_error:
MAE : 0.2339 	 RMSE: 0.2745 
Diagonal-Error %: 19.4108 %

test_error:
MAE : 0.2345 	 RMSE: 0.2785 
Diagonal-Error %: 19.6916 %
Test-Diagonal-Error: 19.6916% | Improvment: 2

[19:40:33] Epoch 1: 1071.67 Sekunden | Running_loss: 0.235 | test_diag_error=19.6916% | train_diag_error=19.4108% | 



Epoch 2: 100%|█████████████████████████████████████████████████████████████████████| 313/313 [07:50<00:00,  1.50s/it, loss=0.267]



train_error:
MAE : 0.2331 	 RMSE: 0.2736 
Diagonal-Error %: 19.3481 %

test_error:
MAE : 0.2348 	 RMSE: 0.2785 
Diagonal-Error %: 19.6917 %
Patience: 1/3

[19:57:25] Epoch 2: 1011.67 Sekunden | Running_loss: 0.234 | test_diag_error=19.6917% | train_diag_error=19.3481% | 



...


train_error:
MAE : 0.2337 	 RMSE: 0.2741 
Diagonal-Error %: 19.3844 %

test_error:
MAE : 0.2345 	 RMSE: 0.2787 
Diagonal-Error %: 19.7062 %
Patience: 1/3

[00:44:04] Epoch 2: 978.02 Sekunden | Running_loss: 0.234 | test_diag_error=19.7062% | train_diag_error=19.3844% | 



Epoch 3: 100%|█████████████████████████████████████████████████████████████████████| 313/313 [07:36<00:00,  1.46s/it, loss=0.213]



train_error:
MAE : 0.2319 	 RMSE: 0.2725 
Diagonal-Error %: 19.2673 %

test_error:
MAE : 0.2349 	 RMSE: 0.2784 
Diagonal-Error %: 19.6882 %
Patience: 2/3

[01:00:20] Epoch 3: 975.94 Sekunden | Running_loss: 0.234 | test_diag_error=19.6882% | train_diag_error=19.2673% | 



Epoch 4: 100%|█████████████████████████████████████████████████████████████████████| 313/313 [07:37<00:00,  1.46s/it, loss=0.228]



train_error:
MAE : 0.2301 	 RMSE: 0.2719 
Diagonal-Error %: 19.2241 %

test_error:
MAE : 0.2356 	 RMSE: 0.2794 
Diagonal-Error %: 19.7576 %
Patience: 3/3

Early Stopping after 4 Epochen.
Imporovments totally: 2

totll running-tiems (s): 4448.34 Sekunden
totll running-tiems (min): 74.14 Minuten

last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 74.14 min	 | improvements: 2 Epochen	 | best_test_error: 19.6562 % 

  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=
  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=


 Parameter:
 best_test_error: [19.69159 19.67359 19.69785 19.69303 19.65619], 
 running_time: [78.83583934 76.00137809 59.62526972 76.32883505 74.13899965] 



	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 10000, 	 test: 2000
	 batch_Size: 	 32
	 def_dataset: 	 norm_subject
	 learning_rate: 	 0.0001
	 epochs: 	 500
	 patience: 	 3

     param in model.layer4.parameters: False
     param in model.fc.parameters: True

# def_dataset_size: train: 10000, 	 test: 2000
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 78.84 min	 | improvements: 2 Epochen	 | best_test_error: 19.6916 % 
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 76.00 min	 | improvements: 2 Epochen	 | best_test_error: 19.6736 %
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 59.63 min	 | improvements: 1 Epochen	 | best_test_error: 19.6978 % 
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 76.33 min	 | improvements: 2 Epochen	 | best_test_error: 19.6930 % 
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 74.14 min	 | improvements: 2 Epochen	 | best_test_error: 19.6562 %

best_test_error =  [19.69159, 19.67359, 19.69785, 19.69303, 19.65619] 
running_time    =  [78.84, 76.00, 59.63, 76.33, 74.14] 


	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 32
	 def_dataset: 	 norm_subject
	 learning_rate: 	 0.0001
	 epochs: 	 500
	 patience: 	 3
     
     param in model.layer4.parameters: False
     param in model.fc.parameters: True
     
# def_dataset_size: train: 1000, 	 test: 200
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 7.06 min	 | improvements: 1 Epochen	 | best_test_error: 20.0895 % 
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 7.91 min	 | improvements: 2 Epochen	 | best_test_error: 20.0419 %
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 7.48 min	 | improvements: 2 Epochen	 | best_test_error: 20.0626 % 
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 5.75 min	 | improvements: 1 Epochen	 | best_test_error: 20.1050 %
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 7.63 min	 | improvements: 2 Epochen	 | best_test_error: 20.0442 % 



best_test_error = [20.08953, 20.04193, 20.06257, 20.10499, 20.04419]
running_time    = [7.06, 7.91, 7.48, 5.75, 7.63]



In [ ]:
# dataset_size: 1000 & 200
best_test_error = [20.08953, 20.04193, 20.06257, 20.10499, 20.04419]
running_time    = [7.06, 7.91, 7.48, 5.75, 7.63]

# dataset_size: 10000 & 2000
best_test_error =  [19.69159, 19.67359, 19.69785, 19.69303, 19.65619] 
running_time    =  [78.84, 76.00, 59.63, 76.33, 74.14] 

In [ ]:
# FC + lay 4:
best_test_error =  [19.95091, 20.2972, 20.07323, 20.04155, 20.19138]
running_time    =  [7.626, 10.104, 7.248, 7.389, 7.081]

# FC only:
best_test_error = [20.08953, 20.04193, 20.06257, 20.10499, 20.04419], 
running_time    = [7.057, 7.908 7.478, 5.753, 7.628]

In [49]:

test_error_parm, run_time_parm = [], []
for i in range(len(parameter)):
    if i % 2 == 0:
        test_error_parm.append((np.round(parameter[i], 5)))
    else:
        run_time_parm.append(np.round(parameter[i], 3))

print(f"\n test_error_parm: {np.float64(test_error_parm)} \n run_time_parm: {np.float64(run_time_parm)}\n\n")


parameter2 = []

parm_test_error = []
parm_run_time = []

for act, values in act_time_epochs.items():
    print(
        f"last_layer: {act}\t"
        f" epochs: {values['epochs']} Epochen\t|"
        f" running_time: {values['time_minutes']:.2f} min\t |"
        f" improvements: {values['improvements']} Epochen\t |"
        f" best_test_error: {values['best_test_error']:.4f} % "
    )
    parameter2.append(float(f"{values['best_test_error']:.4f}"))
    parm_test_error.append(float(f"{values['best_test_error']:.4f}"))
    parm_run_time.append(float(f"{values['time_minutes']:.2f}"))

print(f"\n parameter2: {parameter2}")
print(f"\n parameter: {np.float64([np.float64(parameter[p]) for p in range(len(parameter))])}")
print(f"\n run_time: {parm_run_time}")
print(f"\n test_error: {parm_test_error}")


 test_error_parm: [20.19735 20.17375 19.98697 20.11189 20.06467] 
 run_time_parm: [10.507 11.879  6.817 11.947  6.362]


last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 6.36 min	 | improvements: 1 Epochen	 | best_test_error: 20.0647 % 

 parameter2: [20.0647]

 parameter: [20.19735    10.50747222 20.17375    11.87886853 19.98697     6.81678813
 20.11189    11.94715597 20.06467     6.36239556]

 run_time: [6.36]

 test_error: [20.0647]


    dataset_name: 	  norm_labels.csv
    def_dataset_size: train: 1000, 	 test: 200
    batch_Size: 	  128
    dataset-splits:   norm_subject
    learning_rate: 	  0.0001
    epochs: 	      500
    patience: 	      3

## batch_size(32 vs. 64 vs. 132):
 * dataset_name: 	   norm_labels.csv
 * dataset_size:       train: 1000, 	 test: 200
 * dataset-splits: 	   norm_subject
 * batch_Size: 	       32 vs. 64 vs. 132      <============
 * learning_rate: 	   1e-4

	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 32
	 def_dataset: 	 norm_subject
	 learning_rate: 	 1e-05
	 epochs: 	 500
	 patience: 	 3

# batch_size: 	 32
 
* ** try 1 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 7.44 min	 | improvements: 2 Epochen	 | best_test_error: 20.0913 % 

* ** try 2 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 7.34 min	 | improvements: 2 Epochen	 | best_test_error: 20.0092 %  

* ** try 3 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 6.07 min	 | improvements: 1 Epochen	 | best_test_error: 20.0785 %

* ** try 4 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 8.69 min	 | improvements: 2 Epochen	 | best_test_error: 20.0367 % 

* ** try 5 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 8.65 min	 | improvements: 2 Epochen	 | best_test_error: 20.0945 %

########################################################################
	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 64
	 def_dataset: 	 norm_subject
	 learning_rate: 	 1e-05
	 epochs: 	 500
	 patience: 	 3

# batch_size: 	 64

* ** try 1 **:
last_layer: Sigmoid	 epochs: 5 Epochen	| running_time: 10.26 min	 | improvements: 3 Epochen	 | best_test_error: 20.2225 % 

* ** try 2 **:
last_layer: Sigmoid	 epochs: 7 Epochen	| running_time: 14.27 min	 | improvements: 5 Epochen	 | best_test_error: 20.2962 %

* ** try 3 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 8.27 min	 | improvements: 2 Epochen	 | best_test_error: 20.1555 %

* ** try 4 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 8.08 min	 | improvements: 2 Epochen	 | best_test_error: 19.9577 %

* ** try 5 **:
last_layer: Sigmoid	 epochs: 5 Epochen	| running_time: 10.08 min	 | improvements: 3 Epochen	 | best_test_error: 19.9995 %

########################################################################
	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 128
	 def_dataset: 	 norm_subject
	 learning_rate: 	 1e-05
	 epochs: 	 500
	 patience: 	 3

# batch_size: 	 128
 
* ** try 1 **:
last_layer: Sigmoid	 epochs: 5 Epochen	| running_time: 10.51 min	 | improvements: 3 Epochen	 | best_test_error: 20.1974 %

* ** try 2 **:
last_layer: Sigmoid	 epochs: 6 Epochen	| running_time: 11.88 min	 | improvements: 4 Epochen	 | best_test_error: 20.1737 % 

* ** try 3 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 6.82 min	 | improvements: 1 Epochen	 | best_test_error: 19.9870 %

* ** try 4 **:
last_layer: Sigmoid	 epochs: 6 Epochen	| running_time: 11.95 min	 | improvements: 4 Epochen	 | best_test_error: 20.1119 %

* ** try 5 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 6.36 min	 | improvements: 1 Epochen	 | best_test_error: 20.0647 %


In [51]:
parm_bat32_te = [20.0913, 20.0092, 20.0785, 20.0367, 20.0945]
parm_bat32_rt = [7.44, 7.34, 6.07, 8.69, 8.65]

parm_bat64_te = [20.2225, 20.2962, 20.1555, 19.9577, 19.9995]
parm_bat64_rt = [10.26, 14.27, 8.27, 8.08, 10.08]

parm_bat128_te = [20.19735, 20.17375, 19.98697, 20.11189, 20.06467]
parm_bat128_rt = [10.507, 11.879, 6.817, 11.947, 6.362]


In [59]:
# parm_bat_32: 

# parm_bat32_te = [20.0913, 20.0092, 20.0785, 20.0367, 20.0945]
# parm_bat32_rt = [7.44, 7.34, 6.07, 8.69, 8.65]

parm_test_error   = parm_bat32_te
parm_running_time = parm_bat32_rt


print('\n\n')
print('#'*60)
print('#'*15, "\t Confidence Intervall: ", '#'*15, sep='')
print('#'*60)

conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time: \t\t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)




############################################################
###############	 Confidence Intervall: ###############
############################################################

 Test-Error : 		 Mean = 20.06204, 
	 Confidence interval 0.98% = [19.99926, 20.12482]

 Running-Time: 		 Mean = 7.638, 
	 Confidence interval 0.98%:  [5.818, 9.458]




In [61]:
# parm_bat_128:

# parm_bat128_te = [20.19735, 20.17375, 19.98697, 20.11189, 20.06467]
# parm_bat128_rt = [10.507, 11.879, 6.817, 11.947, 6.362]


parm_test_error   = parm_bat64_te
parm_running_time = parm_bat64_rt

print('\n\n')
print('#'*60)
print('#'*15, "\t Confidence Intervall: ", '#'*15, sep='')
print('#'*60)

conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time: \t\t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)




############################################################
###############	 Confidence Intervall: ###############
############################################################

 Test-Error : 		 Mean = 20.12628, 
	 Confidence interval 0.98% = [19.88421, 20.36835]

 Running-Time: 		 Mean = 10.192, 
	 Confidence interval 0.98%:  [6.019, 14.365]




In [62]:
# parm_bat_128:

parm_test_error   = parm_bat128_te
parm_running_time = parm_bat128_rt
print('\n\n')
print('#'*60)
print('#'*15, "\t Confidence Intervall: ", '#'*15, sep='')
print('#'*60)

conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time: \t\t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)




############################################################
###############	 Confidence Intervall: ###############
############################################################

 Test-Error : 		 Mean = 20.10693, 
	 Confidence interval 0.98% = [19.96464, 20.24922]

 Running-Time: 		 Mean = 9.502, 
	 Confidence interval 0.98%:  [4.936, 14.069]




In [ ]:
parm_bat32_te = [20.0913, 20.0092, 20.0785, 20.0367, 20.0945]
parm_bat32_rt = [7.44, 7.34, 6.07, 8.69, 8.65]

parm_bat64_te = [20.2225, 20.2962, 20.1555, 19.9577, 19.9995]
parm_bat64_rt = [10.26, 14.27, 8.27, 8.08, 10.08]

parm_bat128_te = [20.19735, 20.17375, 19.98697, 20.11189, 20.06467]
parm_bat128_rt = [10.507, 11.879, 6.817, 11.947, 6.362]


# 32:
 Test-Error : 		 Mean = 20.06204, 
	 Confidence interval 0.98% = [19.99926, 20.12482]
# 64:
 Test-Error : 		 Mean = 20.12628, 
	 Confidence interval 0.98% = [19.88421, 20.36835]
# 128:
 Test-Error : 		 Mean = 20.10693, 
	 Confidence interval 0.98% = [19.96464, 20.24922]


# 32:
 Running-Time: 		 Mean = 7.638, 
	 Confidence interval 0.98%:  [5.818, 9.458]
# 64:
 Running-Time: 		 Mean = 10.192, 
	 Confidence interval 0.98%:  [6.019, 14.365]
# 128:
 Running-Time: 		 Mean = 9.502, 
	 Confidence interval 0.98%:  [4.936, 14.069]


## def_dataset(norm_subject, norm_random):
 * dataset_name: 	   norm_labels.csv
 * dataset_size:       train: 1000, 	 test: 200
 * dataset-splits: 	   norm_subject vs. norm_random      <============
 * batch_Size: 	       128
 * learning_rate: 	   1e-4

	 dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 128
	 def_dataset: 	 norm_subject
	 learning_rate: 	 0.0001
	 epochs: 	 500
	 patience: 	 3

# dataset-splits: 	 norm_subject
 
* ** try 1 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.30 min	 | improvements: 2 Epochen	 | best_test_error: 20.1205 % 

* ** try 2 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.25 min	 | improvements: 2 Epochen	 | best_test_error: 20.0771 % 

* ** try 3 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.22 min	 | improvements: 2 Epochen	 | best_test_error: 20.0507 %

* ** try 4 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.04 min	 | improvements: 2 Epochen	 | best_test_error: 20.0485 % 

* ** try 5 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.21 min	 | improvements: 2 Epochen	 | best_test_error: 20.0101 % 

* ** try 6 **:
last_layer: Sigmoid	 epochs: 5 Epochen	| running_time: 11.28 min	 | improvements: 3 Epochen	 | best_test_error: 19.9546 %

parm_norm_sub_te = [20.1205, 20.0771, 20.0507, 20.0485, 20.0101, 19.9546]
parm_norm_sub_rt = [9.30, 9.25, 9.22, 9.04, 9.21, 11.28]


########################################################################
# dataset-splits: 	 norm_random

* ** try 1 **:
last_layer: Sigmoid	 epochs: 3 Epochen	| running_time: 7.24 min	 | improvements: 1 Epochen	 | best_test_error: 18.8797 %

* ** try 2 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.14 min	 | improvements: 2 Epochen	 | best_test_error: 18.8308 %

* ** try 3 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.24 min	 | improvements: 2 Epochen	 | best_test_error: 18.8890 %

* ** try 4 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 8.98 min	 | improvements: 2 Epochen	 | best_test_error: 18.8544 %

* ** try 5 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.72 min	 | improvements: 2 Epochen	 | best_test_error: 18.9156 %

* ** try 6 **:
last_layer: Sigmoid	 epochs: 4 Epochen	| running_time: 9.17 min	 | improvements: 2 Epochen	 | best_test_error: 18.8534 %


parm_norm_random_te = [18.8797, 18.8308, 18.8890, 18.8544, 18.9156, 18.8534]
parm_norm_random_rt = [7.24, 9.14, 9.24, 8.98, 9.72, 9.17]

In [70]:

parm_norm_sub_te = [20.1205, 20.0771, 20.0507, 20.0485, 20.0101, 19.9546]
parm_norm_sub_rt = [9.30, 9.25, 9.22, 9.04, 9.21, 11.28]

parm_norm_random_te = [18.8797, 18.8308, 18.8890, 18.8544, 18.9156, 18.8534]
parm_norm_random_rt = [7.24, 9.14, 9.24, 8.98, 9.72, 9.17]

## dataset_size:((10000, 2000) vs. (1000, 200)):
 * dataset_name: 	   norm_labels.csv
 * dataset_size:       (10000, 2000) vs. (1000, 200)    <============  
 * dataset-splits: 	   norm_subject
 * batch_Size: 	       32
 * learning_rate: 	   1e-4   

# Dataset_size : 10000 & 2000 vs 1000 & 200


In [26]:
# Dataset_size: 10000 & 2000
dataset_size1 = [19.69159, 19.67359, 19.69785, 19.69303, 19.65619]           # 10000: train & 2000: test 
running_time1  = [78.84, 76.00, 59.63, 76.33, 74.14]


# Dataset_size: 1000 & 200
dataset_size2 = [20.08953, 20.04193, 20.06257, 20.10499, 20.04419]           # 1000:  train & 200:  test
running_time2 = [7.06, 7.91, 7.48, 5.75, 7.63]


test_error1 = dataset_size1
test_error2 = dataset_size2

run_time1 = running_time1
run_time2 = running_time2

###############	 Confidence Intervall: ###############

parm_test_error   = test_error1
parm_running_time = run_time1


thema = "Dataset_size = 10000 & 2000"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: \t", '#'*20, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error1 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time1 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)



###############	 Confidence Intervall: ###############

parm_test_error   = test_error2
parm_running_time = run_time2


thema = "Dataset_size = 1000 & 200"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: \t", '#'*20, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error2 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time2 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)




  Thema: 	 Dataset_size = 10000 & 2000 

###############	 Confidence Intervall: 	####################

 Test-Error1 : 		 Mean = 19.68245, 
	 Confidence interval 0.98% = [19.65342, 19.71148]

 Running-Time1 : 	 Mean = 72.988, 
	 Confidence interval 0.98%:  [60.165, 85.811]




  Thema: 	 Dataset_size = 1000 & 200 

###############	 Confidence Intervall: 	####################

 Test-Error2 : 		 Mean = 20.06864, 
	 Confidence interval 0.98% = [20.02195, 20.11533]

 Running-Time2 : 	 Mean = 7.166, 
	 Confidence interval 0.98%:  [5.743, 8.589]




# trainable model layers: layer 4 & FC vs. only FC

In [18]:
# FC + lay 4:
test_error1  =  [19.95091, 20.2972, 20.07323, 20.04155, 20.19138]
run_time1    =  [7.626, 10.104, 7.248, 7.389, 7.081]

# FC only:
test_error2  = [20.08953, 20.04193, 20.06257, 20.10499, 20.04419]
run_time2    = [7.057, 7.908, 7.478, 5.753, 7.628]


###############	 Confidence Intervall: ###############

parm_test_error   = test_error1
parm_running_time = run_time1


thema = "trainable model-layers: layer 4 & fc"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: \t", '#'*20, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error1 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time1 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}% =  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)



###############	 Confidence Intervall: ###############

parm_test_error   = test_error2
parm_running_time = run_time2


thema = "trainable model-layers: only fc"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: \t", '#'*20, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error2 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time2 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}% =  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)


################### trainable model-layers: layer 4 & fc 
# Test-Error1 : 		 
Mean = 20.11085, 
Confidence interval 0.98% = [19.88447, 20.33724]

# Running-Time1 : 	 
Mean = 7.890, 
Confidence interval 0.98% =  [5.788, 9.991]


################## trainable model-layers: only fc 
# Test-Error2 : 		 
Mean = 20.06864 
Confidence interval 0.98% = [20.02195, 20.11533]

# Running-Time2 : 	 
Mean = 7.165 
Confidence interval 0.98% =  [5.745, 8.584]



  Thema: 	 trainable model-layers: layer 4 & fc 

###############	 Confidence Intervall: 	####################

 Test-Error1 : 		 Mean = 20.11085, 
	 Confidence interval 0.98% = [19.88447, 20.33724]

 Running-Time1 : 	 Mean = 7.890, 
	 Confidence interval 0.98%:  [5.788, 9.991]




  Thema: 	 trainable model-layers: only fc 

###############	 Confidence Intervall: 	####################

 Test-Error2 : 		 Mean = 20.06864, 
	 Confidence interval 0.98% = [20.02195, 20.11533]

 Running-Time2 : 	 Mean = 7.165, 
	 Confidence interval 0.98%:  [5.745, 8.584]




In [ ]:
 dataset_name: 	 norm_labels.csv

 def_dataset_size: train: 1000, 	 test: 200

 batch_Size: 	 32

 def_dataset: 	 norm_subject

 learning_rate: 	 0.0001

In [ ]:
# loss 1:
test_error1 = [20.0098, 	20.2339, 	20.0127, 	20.2048, 	20.1970]
run_time1 = [8.15, 		8.67, 		9.12, 		15.50, 		10.61]

# loss 2 (loss_sensetive)
test_error2 = [20.1656, 	20.2252, 	20.1940, 	20.1018, 	20.1746]
run_time2 = [14.84, 		9.96, 		13.57, 		11.91, 		8.09]



###############	 Confidence Intervall: ###############

parm_test_error   = test_error1
parm_running_time = run_time1


thema = "loss_func = loss1"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: \t", '#'*20, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error1 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time1 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)



###############	 Confidence Intervall: ###############

parm_test_error   = test_error2
parm_running_time = run_time2


thema = "loss_func = loss2(loss_sensetive)"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: \t", '#'*20, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error2 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time2 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)


############### loss_func = loss1 
# Test-Error1 : 		 
Mean = 20.13164, 
Confidence interval 0.98 % = [19.94604, 20.31724]

# Running-Time1 : 	 
Mean = 10.410
Confidence interval 0.98 % =  [5.401, 15.419]




############### loss_func = loss2(loss_sensetive) 
# Test-Error2 : 		 
Mean = 20.17224 
Confidence interval 0.98% = [20.09596, 20.24852]

# Running-Time2 : 	 
Mean = 11.674
Confidence interval 0.98%:  [7.127, 16.221]

In [90]:
"""     
     dataset_name: 	 norm_labels.csv
	 def_dataset_size: train: 1000, 	 test: 200
	 batch_Size: 	 128
	 def_dataset: 	 norm_subject
	 learning_rate: 	 XXXXXXXXXXXXXXXXXXX
	 epochs: 	 500
	 patience: 	 3

"""

# 1e-3:

best_test_error_3 = [20.08309, 20.07633, 20.14909, 20.1672, 20.12841]
running_time_3    = [6.39,     7.94,     6.29,     6.99,    7.01]

########################################################################
# 1e-4:

best_test_error_4 = [20.03305, 20.00925, 20.0298, 20.04046, 19.99575] 
running_time_4    = [9.125,    9.68,     6.48,    8.37,     6.49] 
########################################################################
# 1e-5:

best_test_error_5 = [20.08512, 20.10841, 20.11687, 20.10085, 20.11486] 
running_time_5    = [8.31,     12.15,    12.055, 6.50,          8.39]

########################################################################
# 1e-6:

best_test_error_6 = [20.32283, 20.0783, 20.05378, 20.33607, 20.36392]
running_time_6    = [6.50,     17.73,   19.66,    8.37,     6.50]


######################################################
###############	 Confidence Intervall: ###############
######################################################

parm_test_error   = best_test_error_3

parm_running_time = running_time_3


thema = "lr = 1e-3"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: ", '#'*15, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error1 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time1 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)


######################################################
###############	 Confidence Intervall: ###############
######################################################

parm_test_error   = best_test_error_4
parm_running_time = running_time_4


thema = "lr = 1e-4"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: ", '#'*15, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error1 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time1 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)


######################################################
###############	 Confidence Intervall: ###############
######################################################

parm_test_error   = best_test_error_5
parm_running_time = running_time_5


thema = "lr = 1e-5"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: ", '#'*15, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error1 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time1 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)


######################################################
###############	 Confidence Intervall: ###############
######################################################

parm_test_error   = best_test_error_6
parm_running_time = running_time_6

thema = "lr = 1e-6"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: ", '#'*15, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error1 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time1 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)

############### lr = 1e-3 
# Test-Error1 : 		 
Mean = 20.12082
Confidence interval 0.98% = [20.05374, 20.18791]

# Running-Time1 : 	 
Mean = 6.924, 
Confidence interval 0.98% =  [5.822, 8.026]


############### lr = 1e-4 
# Test-Error1 : 		 
Mean = 20.02166
Confidence interval 0.98% = [19.99060, 20.05273]

# Running-Time1 : 	 
Mean = 8.029 
Confidence interval 0.98% =  [5.542, 10.516]


############### lr = 1e-5 
# Test-Error1 : 		 
Mean = 20.10522
Confidence interval 0.98% = [20.08367, 20.12678]

# Running-Time1 : 	 
Mean = 9.481 
Confidence interval 0.98% =  [5.275, 13.687]


############### lr = 1e-6 
# Test-Error1 : 		 
Mean = 20.23098
Confidence interval 0.98% = [19.97704, 20.48492]

# Running-Time1 : 	 
Mean = 11.752, 
Confidence interval 0.98% =  [0.994, 22.510]




  Thema: 	 lr = 1e-3 

###############	 Confidence Intervall: ###############

 Test-Error1 : 		 Mean = 20.12082, 
	 Confidence interval 0.98% = [20.05374, 20.18791]

 Running-Time1 : 	 Mean = 6.924, 
	 Confidence interval 0.98%:  [5.822, 8.026]




  Thema: 	 lr = 1e-4 

###############	 Confidence Intervall: ###############

 Test-Error1 : 		 Mean = 20.02166, 
	 Confidence interval 0.98% = [19.99060, 20.05273]

 Running-Time1 : 	 Mean = 8.029, 
	 Confidence interval 0.98%:  [5.542, 10.516]




  Thema: 	 lr = 1e-5 

###############	 Confidence Intervall: ###############

 Test-Error1 : 		 Mean = 20.10522, 
	 Confidence interval 0.98% = [20.08367, 20.12678]

 Running-Time1 : 	 Mean = 9.481, 
	 Confidence interval 0.98%:  [5.275, 13.687]




  Thema: 	 lr = 1e-6 

###############	 Confidence Intervall: ###############

 Test-Error1 : 		 Mean = 20.23098, 
	 Confidence interval 0.98% = [19.97704, 20.48492]

 Running-Time1 : 	 Mean = 11.752, 
	 Confidence interval 0.98%:  [0.994, 22.510

In [92]:
# Splits-type: norm_random vs. norm_subject:


parm_norm_sub_te = [20.1205, 20.0771, 20.0507, 20.0485, 20.0101, 19.9546]
parm_norm_sub_rt = [9.30, 9.25, 9.22, 9.04, 9.21, 11.28]

parm_norm_random_te = [18.8797, 18.8308, 18.8890, 18.8544, 18.9156, 18.8534]
parm_norm_random_rt = [7.24, 9.14, 9.24, 8.98, 9.72, 9.17]


###############	 Confidence Intervall: ###############

parm_test_error   = parm_norm_sub_te
parm_running_time = parm_norm_sub_rt


thema = "Splits-type: subjectindipended"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: \t", '#'*20, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error1 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time1 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)



###############	 Confidence Intervall: ###############

parm_test_error   = parm_norm_random_te
parm_running_time = parm_norm_random_rt


thema = "Splits-type: norm_random"
print(f'\n\n  Thema: \t {thema} \n')

print('#'*15, "\t Confidence Intervall: \t", '#'*20, sep='')


conf = 0.98

test_error_confidence = confidence_interval(parm_test_error, conf)
run_time_confidence   = confidence_interval(parm_running_time, conf)

print(
    f"\n Test-Error2 : \t\t Mean = {test_error_confidence[0]:.5f}, "
    f"\n\t Confidence interval {conf}% = [{test_error_confidence[1]:.5f}, {test_error_confidence[2]:.5f}]"
)

print(
    f"\n Running-Time2 : \t Mean = {run_time_confidence[0]:.3f}, "
    f"\n\t Confidence interval {conf}%:  [{run_time_confidence[1]:.3f}, {run_time_confidence[2]:.3f}]\n\n"
)


############### Splits-type: subjectindipended 
# Test-Error1 : 		 
Mean = 20.04358 
Confidence interval 0.98% = [19.96550, 20.12167]

# Running-Time1 : 	 
Mean = 9.550 
Confidence interval 0.98% =  [8.380, 10.720]



############### Splits-type: norm_random 
# Test-Error2 : 		 
Mean = 18.87048
Confidence interval 0.98% = [18.82887, 18.91210]

# Running-Time2 : 	 
Mean = 8.915 
Confidence interval 0.98% =  [7.737, 10.093] 



  Thema: 	 Splits-type: subjectindipended 

###############	 Confidence Intervall: 	####################

 Test-Error1 : 		 Mean = 20.04358, 
	 Confidence interval 0.98% = [19.96550, 20.12167]

 Running-Time1 : 	 Mean = 9.550, 
	 Confidence interval 0.98%:  [8.380, 10.720]




  Thema: 	 Splits-type: norm_random 

###############	 Confidence Intervall: 	####################

 Test-Error2 : 		 Mean = 18.87048, 
	 Confidence interval 0.98% = [18.82887, 18.91210]

 Running-Time2 : 	 Mean = 8.915, 
	 Confidence interval 0.98%:  [7.737, 10.093]


